In [1]:
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.ops import unary_union
from rasterstats import zonal_stats
import os

Setup and Projections

In [2]:
# We use EPSG:26917 (UTM 17N) because it uses METERS. 
# Buffering in degrees (4326) will not work.
CRS_PROJ = "EPSG:26917"

print("Loading and reprojecting data...")
wv_boundary = gpd.read_file("../data/processed/wv_boundary.gpkg").to_crs(CRS_PROJ)
wv_counties = gpd.read_file("../data/processed/wv_counties.gpkg").to_crs(CRS_PROJ)
roads = gpd.read_file("../data/processed/wv_roads.gpkg").to_crs(CRS_PROJ)

Loading and reprojecting data...


Create the 'Covered' Area

In [3]:
# We assume fiber exists within 2km of major roads
major_road_types = ["motorway", "trunk", "primary", "secondary", "tertiary"]
major_roads = roads[roads["highway"].isin(major_road_types)].copy()

print("Buffering major roads (2km radius)...")
# This creates a 'sausage' shape around every major road
road_buffer = major_roads.geometry.buffer(2000) 
covered_area = unary_union(road_buffer)

Buffering major roads (2km radius)...


Subtract to find the Gaps

In [4]:
print("Identifying gap polygons (State Area minus Covered Area)...")
wv_geom = unary_union(wv_boundary.geometry)
gap_geom = wv_geom.difference(covered_area)

# Explode multi-polygons into individual shapes
gap_gdf = gpd.GeoDataFrame(geometry=[gap_geom], crs=CRS_PROJ).explode(index_parts=True).reset_index(drop=True)

# Filter out tiny slivers (smaller than 1 sq km) to keep the model clean
gap_gdf["area_sqkm"] = gap_gdf.geometry.area / 1_000_000
gap_gdf = gap_gdf[gap_gdf["area_sqkm"] > 1.0].copy()
print(f"Found {len(gap_gdf)} significant gap zones.")

Identifying gap polygons (State Area minus Covered Area)...
Found 376 significant gap zones.


Feature Engineering for Gaps

In [7]:
print("Extracting population and distance features for each gap...")

# A. Population (Zonal Stats) - Must convert to 4326 for WorldPop
gap_gdf_4326 = gap_gdf.to_crs(epsg=4326)
pop_stats = zonal_stats(gap_gdf_4326, "../data/raw/usa_pop_1km.tif", stats="sum")
gap_gdf["pop_count"] = [s['sum'] if s['sum'] is not None else 0 for s in pop_stats]

# B. Distance to Road (Cost proxy)
# Find distance from gap center to the nearest major road
road_union = unary_union(major_roads.geometry)
gap_gdf["dist_to_road_m"] = gap_gdf.geometry.centroid.distance(road_union)

# C. Compactness (Efficiency proxy)
gap_gdf["perimeter"] = gap_gdf.geometry.length
gap_gdf["compactness"] = (4 * np.pi * gap_gdf.geometry.area) / (gap_gdf["perimeter"]**2)

# D Cleaned: County Attribution
print("Attributing gaps to counties...")

# 1. Perform the join
joined = gpd.sjoin(gap_gdf, wv_counties[["NAME", "geometry"]], how="left", predicate="intersects")

# 2. Fix Duplicates: Group by the original gap index and pick the first county name
# This ensures we have exactly one county name per gap row
county_names = joined.groupby(joined.index)['NAME'].first()

# 3. Assign the cleaned names back
gap_gdf["county"] = county_names
gap_gdf["county"] = gap_gdf["county"].fillna("Remote/Border Area")

print("County attribution complete. No duplicates remaining.")

Extracting population and distance features for each gap...
Attributing gaps to counties...
County attribution complete. No duplicates remaining.


The Heuristic Score

Logic: We are going to move away from county-wide averages and look at the Gaps themselves. We will score each gap based on three factors:
- Revenue Potential: Population living inside the gap.
- Infrastructure Cost: Distance from the gap center to the nearest highway.
- Efficiency: How compact the gap is (smaller, denser gaps are cheaper to wire).

The Weighted Formula:
$$Score = (Pop \times 0.5) + (1/Dist \times 0.3) + (Compactness \times 0.2)$$

In [9]:
print("Calculating final opportunity scores...")

def normalize(col):
    return (col - col.min()) / (col.max() - col.min()) if (col.max() - col.min()) != 0 else 0

gap_gdf["n_pop"] = normalize(gap_gdf["pop_count"])
gap_gdf["n_dist"] = 1 - normalize(gap_gdf["dist_to_road_m"]) # Closer is better
gap_gdf["n_comp"] = normalize(gap_gdf["compactness"])

# Weighted Score: 50% Population, 30% Proximity, 20% Compactness
gap_gdf["opportunity_score"] = (gap_gdf["n_pop"] * 0.5) + (gap_gdf["n_dist"] * 0.3) + (gap_gdf["n_comp"] * 0.2)
print("Opportunity scores calculated.")

Calculating final opportunity scores...
Opportunity scores calculated.


Output 

In [11]:
top_10 = gap_gdf.sort_values("opportunity_score", ascending=False).head(10)

print("\n=== TOP 10 BROADBAND EXPANSION TARGETS ===")
print(top_10[["county", "pop_count", "area_sqkm", "dist_to_road_m", "opportunity_score"]].to_string(index=False))

# Save results
os.makedirs("../outputs", exist_ok=True)
top_10.to_crs(epsg=4326).to_file("../outputs/top_10_gaps.gpkg", driver="GPKG")
print("\nSuccess! Top 10 map saved to ../outputs/top_10_gaps.gpkg")


=== TOP 10 BROADBAND EXPANSION TARGETS ===
    county   pop_count  area_sqkm  dist_to_road_m  opportunity_score
   Kanawha 1902.489746 314.827967     5153.901662           0.673922
Monongalia  959.407410  10.788440     2903.863425           0.604141
    Putnam 1331.342285 193.626496     4436.299272           0.588762
    Cabell  702.545715  29.537945     4353.941035           0.539595
  Berkeley  754.474243  38.789640      667.883196           0.517072
   Preston  992.133423 131.760911     4625.033544           0.503553
    Mercer 1074.760864 174.957139     6813.084080           0.483478
    Putnam  391.652283   8.039112     3269.707114           0.481635
   Raleigh  427.111572   6.303963     2904.576878           0.476775
      Wood  526.733887  25.389110      662.507094           0.472993

Success! Top 10 map saved to ../outputs/top_10_gaps.gpkg


ML Model

In [13]:
print(wv_counties.columns.tolist())

['STATEFP', 'COUNTYFP', 'COUNTYNS', 'GEOID', 'GEOIDFQ', 'NAME', 'NAMELSAD', 'LSAD', 'CLASSFP', 'MTFCC', 'CSAFP', 'CBSAFP', 'METDIVFP', 'FUNCSTAT', 'ALAND', 'AWATER', 'INTPTLAT', 'INTPTLON', 'geometry']


In [ ]:
import pandas as pd

# 1. Load the FCC data (Update the path if your file is named differently)
fcc_raw = pd.read_csv("../data/processed/fcc_wv_broadband.csv")

# 2. Convert the 15-digit block ID to a 5-digit County FIPS
# The first 5 digits of a Census Block ID are the County ID
fcc_raw['county_fips'] = fcc_raw['block_geoid'].astype(str).str.zfill(15).str[:5]

# 3. Define 'Served' as 100 Mbps download or better
fcc_raw['is_served'] = fcc_raw['max_advertised_download_speed'] >= 100

# 4. Create the 'county_stats' summary table
# We take the mean of 'is_served' to get a percentage (0.0 to 1.0)
county_stats = fcc_raw.groupby('county_fips')['is_served'].mean().reset_index()
county_stats.columns = ['county_fips', 'broadband_serve_pct']

print("Successfully created 'county_stats'!")
print(f"Sample data:\n{county_stats.head()}")

In [15]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
import pandas as pd
from rasterstats import zonal_stats

print("=== Phase 1: Reconstructing Features ===")

# 1. Add Population (using the WorldPop raster)
# We convert to 4326 because WorldPop is usually in WGS84
stats = zonal_stats(wv_counties.to_crs(epsg=4326), "../data/raw/usa_pop_1km.tif", stats="sum")
wv_counties['total_pop'] = [s['sum'] if s['sum'] is not None else 0 for s in stats]

# 2. Add Road Density
# Assuming roads are already loaded in your notebook as 'roads'
roads_in_county = gpd.sjoin(roads, wv_counties, predicate='within')
road_lengths = roads_in_county.groupby('NAME')['length'].sum()
wv_counties['road_density'] = wv_counties['NAME'].map(road_lengths).fillna(0)

# 3. Add Distance to Backbone
backbone = roads[roads['highway'].isin(['motorway', 'trunk', 'primary'])]
wv_counties['dist_to_backbone_m'] = wv_counties.geometry.centroid.apply(
    lambda x: backbone.distance(x).min()
)

# 4. Add FCC Broadband Stats (Assuming 'county_stats' exists from your FCC processing)
# If county_stats isn't in memory, make sure to run your FCC aggregation block first!
wv_counties = wv_counties.merge(county_stats, left_on="GEOID", right_on="county_fips", how="inner")

print("=== Phase 2: Training Random Forest ===")

# 1. Prepare the Data
features = ['total_pop', 'road_density', 'dist_to_backbone_m']
X = wv_counties[features]
y = wv_counties['broadband_serve_pct']

# 2. Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Train the Model
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 4. Score and Visualize
predictions = model.predict(X_test)
r2 = r2_score(y_test, predictions)

print(f"Model R-squared Score: {r2:.2f}")

# 5. Feature Importance - The "Why" behind the map
importances = model.feature_importances_
feat_imp = pd.Series(importances, index=features).sort_values()
feat_imp.plot(kind='barh', color='teal')
plt.title("Which factor drives Broadband Gaps?")
plt.show()

=== Phase 1: Reconstructing Features ===


NameError: name 'county_stats' is not defined

Feature Importance

In [ ]:
# Extract importance
importances = model.feature_importances_
feature_importance_df = pd.DataFrame({'Feature': features, 'Importance': importances})
feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)

# Plot it
plt.figure(figsize=(8, 5))
plt.barh(feature_importance_df['Feature'], feature_importance_df['Importance'], color='teal')
plt.title("What Drives Broadband Access in WV? (ML Insights)")
plt.xlabel("Importance Score")
plt.show()